# Gold v2 dev — 질문별 독립 검증 agent (설계·선택)

dev 세트(240장 / Q1·Q2 480문항, supervised train)에서 검증 agent 설정을 고릅니다. 고른 설정 하나를 `frozen_config.json`으로 고정·게시한 뒤, gold v2에는 `lvlm_gold_v2_verify_agent_gold_colab.ipynb`로 **한 번만** 돌립니다.

**독립 원칙**: Q1·Q2(·Q3)는 각각 따로 재확인합니다. 한 문항을 재확인할 때는 이미지, 그 문항, 그 문항 자신의 1단계 답만 씁니다. 다른 문항의 답은 프롬프트·재확인 대상 선정·도구 선택·최종 답 어디에도 쓰지 않고, 다른 문항의 답을 바꾸지도 않습니다. 모순은 최종 답을 모두 저장한 뒤 지표로만 계산합니다.

- **1단계**: dev 기준선의 `normal_ref_retrieved_diffdate` 답 재사용 (같은 날 촬영 사진을 뺀 비슷한 정상 이미지, 새 추론 없음)
- **도구** (재확인 때 보는 것 = [정상 참조, target] + 그 문항 + 자기 1단계 답, 그리고)
  - `reask` 새 근거 없음 (다시 묻기 효과 대조군) / `zoom` 가운데 확대 (Q3에는 적용 안 함) / `define` 그 문항 보기의 정의 문장 / `zoom_define` 둘 다
- **경로** (어떤 문항을 재확인할지, 그 문항 자신의 답만 보고 결정)
  - `all` 모든 문항 / `disagreement` 그 문항의 1단계 답 ≠ 같은 문항의 원본만(image_only) 답
- 재확인 답이 유효하면 그 문항의 최종 답, 무효면 1단계 답 유지. 1단계 답과 최종 답 모두 `answers.csv`에 보존
- 보고: Q1 F1 · Q2 macro-F1 · 문항별 정확도, 모순(새로 생김 / 사라짐), Joint, FDR, 재확인 수, 문항당 호출 수, 지연 시간, 답 변경, 1단계 대비 paired bootstrap CI
- 선택 기준(사전 고정): 두 모델 평균 (Q1 F1 + Q2 macro-F1) / 2 변화가 가장 큰 설정, 동률이면 호출 수가 적은 것. 모순은 보고만 하고 점수에 넣지 않음

먼저 `lvlm_gold_v2_dev_baselines_colab.ipynb`의 `normal_ref_retrieved_diffdate`·`image_only` 결과가 Drive에 있어야 합니다.
GPU: **L4**. 호출 수 480문항 × 4도구 × 2모델 = 3,840회(`disagreement` 경로는 `all`의 재확인 답을 같이 써서 추가 호출 없음), 대략 1.5~2시간. 20문항마다 Drive 저장, 끊기면 실행 셀만 다시 실행하면 이어집니다.
결과: `MyDrive/lvlm_insplad_results/gold_v2_verify_agent_dev/` → GitHub `results/latest/gold_v2_verify_agent_dev/`

In [ ]:
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"
DEV_FOLDER = "lvlm_insplad_qa/gold_v2_dev"
DEV_BASELINES_FOLDER = "lvlm_insplad_results/gold_v2_dev_baselines"
RUN_FOLDER = "lvlm_insplad_results/gold_v2_verify_agent_dev"
MODELS = ["internvl3_5_8b","qwen3_5_4b"]
STAGE1_CONDITION = "normal_ref_retrieved_diffdate"
RULE = "diffdate"
TOOLS = ["reask","zoom","define","zoom_define"]
ROUTES = ["all","disagreement"]
CHECKPOINT_EVERY = 20
PUBLISH = True

UNSUP_ZIP_NAME = "unsupervised_anomaly_detection.zip"
UNSUP_ZIP_SIZE = 1_170_967_719
UNSUP_ZIP_PATH = ""   # 비우면 자동 검색
SUP_ZIP_NAME = "supervised_fault_classification.zip"
SUP_ZIP_SIZE = 874_753_167
SUP_ZIP_PATH = ""     # 비우면 자동 검색

In [ ]:
# Drive 연결 + repo clone + 패키지
import base64, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
DEV_DIR = DRIVE_ROOT / DEV_FOLDER
RUN_ROOT = DRIVE_ROOT / RUN_FOLDER
RUN_ROOT.mkdir(parents=True, exist_ok=True)
DEV_REFS_DIR = DEV_DIR / "refs"
STAGE1_DIR = DRIVE_ROOT / DEV_BASELINES_FOLDER / "predictions"

REPO_DIR = Path("/content/lvlm_insplad")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

def run(cmd, *, env=None, check=True):
    """Run a command and stream its output into this cell (Colab hides plain subprocess output)."""
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:12]) + (" ..." if len(cmd) > 12 else ""), flush=True)
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-60:]
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode:
        raise RuntimeError(f"command failed with exit code {p.returncode} (마지막 출력 위 참고)")
    return p.returncode

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 필요합니다.")
os.environ["GITHUB_TOKEN"] = github_token
auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({"GIT_CONFIG_COUNT":"1", "GIT_CONFIG_KEY_0":"http.extraHeader",
                "GIT_CONFIG_VALUE_0":f"AUTHORIZATION: basic {auth}", "GIT_TERMINAL_PROMPT":"0"})
run(["git","clone","--depth","1","--branch",REPO_REF,f"https://github.com/{REPO}.git",REPO_DIR], env=git_env)
del auth, git_env
run([sys.executable,"-m","pip","install","-q","transformers==5.14.0","accelerate==1.14.0",
     "sentencepiece==0.2.2","pandas>=2.2","numpy>=1.26","pillow>=10","tqdm>=4.66","timm"])
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR / "scripts"))
import pandas as pd
import torch
print("repo:", subprocess.check_output(["git","rev-parse","HEAD"], cwd=REPO_DIR, text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
if not torch.cuda.is_available():
    raise RuntimeError("GPU Runtime(L4/A100)을 사용하세요.")

In [ ]:
def find_zip(filename, expected_size, override=""):
    candidates = [Path(override)] if override else []
    candidates += [DRIVE_ROOT / filename, DRIVE_ROOT / "lvlm_insplad_data" / filename]
    for p in candidates:
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    print(f"{filename}: Drive 전체 검색 중 (몇 분 걸릴 수 있음)", flush=True)
    for p in DRIVE_ROOT.rglob(filename):
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    raise FileNotFoundError(f"{filename} ({expected_size} bytes) not found")

SUP_ZIP = find_zip(SUP_ZIP_NAME, SUP_ZIP_SIZE, SUP_ZIP_PATH)
UNSUP_ZIP = find_zip(UNSUP_ZIP_NAME, UNSUP_ZIP_SIZE, UNSUP_ZIP_PATH)
print("SUP:", SUP_ZIP); print("UNSUP:", UNSUP_ZIP)

summary = json.loads((DEV_DIR / "build_summary.json").read_text(encoding="utf-8"))
if summary.get("quality_status") != "FROZEN":
    raise RuntimeError(f"dev 세트가 FROZEN이 아닙니다: {summary.get('quality_status')}")
DEV_QA = DEV_DIR / summary["file"]
REFS_CSV = DEV_REFS_DIR / "retrieved_refs.csv"
need = [DEV_QA, REFS_CSV] + [STAGE1_DIR / m / f"{STAGE1_CONDITION}.jsonl" for m in MODELS] \
       + ([STAGE1_DIR / m / "image_only.jsonl" for m in MODELS] if "disagreement" in ROUTES else [])
missing = [str(p) for p in need if not p.is_file()]
if missing:
    raise FileNotFoundError("먼저 dev 기준선 노트북(diffdate 조건 포함)을 끝내세요. 없음: " + ", ".join(missing))
n_q = summary["questions"]
print(f"입력 확인 완료 | 예상 호출 최대 {n_q * len(TOOLS) * len(MODELS)}회 (이미 저장된 재확인은 재사용)")

In [ ]:
# 실행 — 끊기면 이 셀만 다시 실행해도 완료된 호출은 재사용
run([sys.executable, REPO_DIR / "scripts/run_gold_v2_verify_agent.py",
     "--qa-file", DEV_QA, "--stage1-dir", STAGE1_DIR, "--stage1-condition", STAGE1_CONDITION,
     "--refs-csv", REFS_CSV, "--rule", RULE, "--sup-zip", SUP_ZIP, "--unsup-zip", UNSUP_ZIP,
     "--out-dir", RUN_ROOT, "--models", *MODELS, "--tools", *TOOLS, "--routes", *ROUTES,
     "--checkpoint-every", str(CHECKPOINT_EVERY)])

In [ ]:
# 결과
pd.set_option("display.width", 240); pd.set_option("display.max_rows", 200)
print(json.dumps(json.loads((RUN_ROOT / "route_stats.json").read_text(encoding="utf-8")), ensure_ascii=False, indent=1))
lb = pd.read_csv(RUN_ROOT / "leaderboard.csv")
print("\n[정확도] 문항별로 따로 재확인한 결과")
display(lb[["model_key","variant","q1_anomaly_f1","q2_defect_macro_f1","q1_accuracy","q2_accuracy",
            "rechecks","avg_lvlm_calls_per_q","recheck_latency_s"]].round(3))
print("\n[모순] 최종 답에서 따로 계산 (재확인·답 수정에는 쓰지 않음)")
display(lb[["model_key","variant","consistency_error","contradictions","q1q2_contradictions_new",
            "q1q2_contradictions_resolved","joint_accuracy","false_defect_rate"]].round(3))
print("\n[문항별] 재확인 수 / 바뀐 수 / 맞게 바꾼 수 / 틀리게 바꾼 수")
display(pd.read_csv(RUN_ROOT / "per_question.csv").round(3))
sig = pd.read_csv(RUN_ROOT / "significance.csv")
sig["txt"] = sig.apply(lambda r: f"{r.delta:+.3f} [{r.ci_low:+.3f},{r.ci_high:+.3f}]" + ("*" if r.significant else ""), axis=1)
print("\n[1단계 대비] * = CI가 0을 포함하지 않음 (모순·FDR은 낮을수록 좋음)")
display(sig.pivot_table(index=["model_key","condition"], columns="metric", values="txt", aggfunc="first"))
selection = pd.read_csv(RUN_ROOT / "selection.csv")
print("\n[선택 순위] 사전 고정 기준:", json.loads((RUN_ROOT / "run_metadata.json").read_text(encoding="utf-8"))["selection_criterion"])
display(selection)
status = json.loads((RUN_ROOT / "run_status.json").read_text(encoding="utf-8"))
print("row_errors 합계:", sum(s["row_errors"] for s in status), "(0이 아니면 실행 셀을 다시 실행)")

## 설정 고정

기본은 위 선택 순위 1위입니다. 다른 설정을 쓰려면 gold를 보기 **전에** `SELECTED_VARIANT`를 정하세요. 1단계(검증 없음)보다 나은 설정이 없으면 그 사실도 gold 결과와 함께 그대로 보고합니다.
이 파일이 GitHub에 게시된 뒤에 gold 노트북을 돌립니다 (gold 노트북은 게시된 파일만 읽음).

In [ ]:
SELECTED_VARIANT = None   # 예: "define@all". None이면 선택 순위 1위
import run_gold_v2_verify_agent as agent
dev_meta = json.loads((RUN_ROOT / "run_metadata.json").read_text(encoding="utf-8"))
ranked = selection.to_dict("records")
variant = SELECTED_VARIANT or ranked[0]["variant"]
frozen = agent.make_frozen_config(dev_meta, variant, ranked)
(RUN_ROOT / "frozen_config.json").write_text(json.dumps(frozen, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(json.dumps(frozen, ensure_ascii=False, indent=2))

In [ ]:
if PUBLISH:
    files = ["leaderboard.csv", "per_question.csv", "metrics_by_asset.csv", "answers.csv", "significance.csv",
             "selection.csv", "route_stats.json", "run_status.json", "run_metadata.json"]
    if (RUN_ROOT / "frozen_config.json").is_file():
        files.append("frozen_config.json")
    run([sys.executable, REPO_DIR / "scripts/publish_results_to_github.py", "--out-dir", RUN_ROOT,
         "--repo", REPO, "--branch", REPO_REF, "--prefix", "gold_v2_verify_agent_dev", "--files", *files],
        env=os.environ.copy())
    print("GitHub: results/latest/gold_v2_verify_agent_dev/")